# NB27 — Weighted F1 + Threshold Optimization + Missing Robustness + Label Noise

**TEKNOFEST 2026 | Genetik Varyant Patojenite Tahmini**

## Amaç
to-do.md'deki GÖREV 1-4'ü uygula:

1. **Ağırlıklı F1 değerlendirme sistemi** (deterministik, tüm veriyi kullanır)
2. **Threshold düzeltmesi** (ağırlıklı F1-max ile)
3. **Label noise tespiti** (cross-model consensus + cleanlab — PAH)
4. **Missing value robustness** (4 strateji: M3 baseline, flagsiz, KNN, sentinel)

## Baseline'lar
- **PAH**: NB21 P4_COMBINED_BalBag → Boot %80/20 F1=0.582, MCC=0.529, thr=select_threshold_8020_robust
- **CFTR**: NB20 S0c_COMBINED → Boot %80/20 F1=0.863, FP=0, thr=0.1474

## Sonuç
Her GÖREV'in sonuçları ayrı CSV'ler ve görseller oluşturur. Toplam: 4 CSV + 4 PNG.

# Cell 1: Imports & Config

In [1]:
import os, sys, warnings, json
from datetime import datetime
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR
from src.metrics import compute_all_metrics, optimize_threshold

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, LeaveOneOut
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, matthews_corrcoef, precision_score, recall_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.impute import KNNImputer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedBaggingClassifier

# Optional: cleanlab (label noise detection)
try:
    from cleanlab.filter import find_label_issues
    HAS_CLEANLAB = True
except ImportError:
    HAS_CLEANLAB = False
    print("cleanlab not installed (pip install cleanlab for label noise detection)")

np.random.seed(SEED)

# Constants
PI_TEST = 0.20                          # final test: 20% pathogenic
FINAL_BENIGN_FRAC = 0.80                # final test: 80% benign
N_BOOT = 50                             # bootstrap samples
BOOT_SEED = 123                         # reproducibility
HIGH_MISS_THR = 0.50                    # high-missing threshold

# Results directories
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v12_optimization")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"NB27 — Weighted F1 + Threshold Optimization + Missing Robustness + Label Noise")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")

NB27 — Weighted F1 + Threshold Optimization + Missing Robustness + Label Noise
SEED=42, PI_TEST=0.2, N_BOOT=50
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization


# Cell 2: Data Loading + Column Cleanup

In [2]:
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

# --- Load all 4 datasets ---
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# --- Create combined training pools ---
# For PAH experiments: MASTER + KANSER + CFTR
df_combined_pah = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
# For CFTR experiments: MASTER + KANSER + PAH
df_combined_cftr = pd.concat([df_master, df_kanser, df_pah], ignore_index=True)

print(f"\nCOMBINED_PAH: {df_combined_pah.shape} (pos={df_combined_pah[TARGET].sum()})")
print(f"COMBINED_CFTR: {df_combined_cftr.shape} (pos={df_combined_cftr[TARGET].sum()})")

# --- Remove cross-panel exact duplicates (NB21 pattern) ---
feat_cols_all = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Find all-column-identical rows between panel and master."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_rows = panel_df.loc[panel_df[ID_COL] == vid, check_cols]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, p_row in p_rows.iterrows():
            for _, m_row in m_rows.iterrows():
                if p_row.equals(m_row):
                    dup_ids.append(vid)
                    break
    return dup_ids

# Remove PAH duplicates
dup_ids_pah = find_exact_dups(df_pah, df_master, feat_cols_all, TARGET)
if dup_ids_pah:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids_pah)].reset_index(drop=True)
    print(f"PAH: dropped {len(dup_ids_pah)} exact duplicates -> {df_pah.shape}")

# Remove CFTR duplicates
dup_ids_cftr = find_exact_dups(df_cftr, df_master, feat_cols_all, TARGET)
if dup_ids_cftr:
    df_cftr = df_cftr[~df_cftr[ID_COL].isin(dup_ids_cftr)].reset_index(drop=True)
    print(f"CFTR: dropped {len(dup_ids_cftr)} exact duplicates -> {df_cftr.shape}")

# --- Column cleanup: constant & duplicate columns ---
def get_constant_cols(df, cols):
    """Find columns with nunique <= 1."""
    return [c for c in cols if df[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    """Find identical column pairs."""
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

# Detect on MASTER
const_cols = get_constant_cols(df_master, feat_cols_all)
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, feat_cols_all)
drop_cols = set(const_cols) | dup_drop

print(f"\nColumn cleanup: {len(const_cols)} constant + {len(dup_drop)} duplicate -> drop {len(drop_cols)}")
print(f"Remaining features: {len(feat_cols_all) - len(drop_cols)}")

# Apply to all datasets
keep_cols = [c for c in feat_cols_all if c not in drop_cols]
for df in [df_master, df_kanser, df_cftr, df_pah]:
    cols_to_keep = [ID_COL, TARGET] + keep_cols
    for col in df.columns:
        if col not in cols_to_keep:
            df.drop(col, axis=1, inplace=True)

# Recreate combined datasets
df_combined_pah = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
df_combined_cftr = pd.concat([df_master, df_kanser, df_pah], ignore_index=True)

print(f"\nFinal shapes:")
print(f"  MASTER: {df_master.shape}")
print(f"  KANSER: {df_kanser.shape}")
print(f"  CFTR:   {df_cftr.shape}")
print(f"  PAH:    {df_pah.shape}")
print(f"  COMBINED_PAH: {df_combined_pah.shape}")
print(f"  COMBINED_CFTR: {df_combined_cftr.shape}")

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353)   (pos=90, neg=21)
PAH:    (372, 353)  (pos=310, neg=62)

COMBINED_PAH: (3430, 353) (pos=2507)
COMBINED_CFTR: (3691, 353) (pos=2727)
PAH: dropped 3 exact duplicates -> (369, 353)

Column cleanup: 0 constant + 58 duplicate -> drop 58
Remaining features: 293

Final shapes:
  MASTER: (2931, 295)
  KANSER: (388, 295)
  CFTR:   (111, 295)
  PAH:    (369, 295)
  COMBINED_PAH: (3430, 295)
  COMBINED_CFTR: (3688, 295)


# Cell 3: FE + M3 Preprocessing (with multiple strategies)

In [3]:
AA_UNK = CR.AA_UNKNOWN_TOKEN

def fit_preprocessor(train_df, keep_cols, target, strategy="m3"):
    """
    Fit preprocessor: median/mode, label encoder, high-missing detection.
    Strategies:
      - m3: is_missing flags (>50% NaN) + median imputation
      - no_flags: no is_missing flags, just median imputation
      - knn: KNN imputation (k=5)
      - sentinel: use -999 sentinel for missing (tree models)
    """
    X = train_df[keep_cols].copy()
    
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing columns (>50% NaN)
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    # Median imputation values
    medians = X[num_cols].median()
    
    # Label encoders for categoricals
    le_maps = {}
    for c in cat_cols:
        le = LabelEncoder()
        vals = X[c].fillna("MISSING").astype(str).unique()
        le.fit(vals)
        le_maps[c] = le
    
    # KNN imputer (fit on numeric cols)
    knn_imputer = None
    if strategy == "knn":
        knn_imputer = KNNImputer(n_neighbors=5, weights="distance")
        knn_imputer.fit(X[num_cols].values)
    
    return {
        "strategy": strategy,
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "high_miss": high_miss,
        "medians": medians,
        "le_maps": le_maps,
        "knn_imputer": knn_imputer
    }

def transform_X(df, keep_cols, prep):
    """
    Apply preprocessor: imputation, encoding, missing flags.
    """
    X = df[keep_cols].copy()
    strategy = prep["strategy"]
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # === Numeric imputation ===
    if strategy == "m3" or strategy == "no_flags":
        # Median imputation
        for c in num_cols:
            X[c] = X[c].fillna(prep["medians"][c])
    elif strategy == "knn":
        # KNN imputation
        X_num = prep["knn_imputer"].transform(X[num_cols].values)
        X[num_cols] = X_num
    elif strategy == "sentinel":
        # Sentinel value -999 for tree models
        for c in num_cols:
            X[c] = X[c].fillna(-999.0)
    
    # === is_missing flags (only for m3) ===
    if strategy == "m3":
        for c in prep["high_miss"]:
            X[f"is_missing_{c}"] = X[c].isnull().astype(int)
        # Fill after flag creation
        for c in num_cols:
            X[c] = X[c].fillna(prep["medians"][c])
    
    # === Categorical encoding ===
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING").astype(str)
        le = prep["le_maps"][c]
        # Unseen categories -> -1
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    # Add missing_count feature
    X["missing_count"] = df[keep_cols].isnull().sum(axis=1)
    
    return X

# Fit preprocessors for each strategy (on combined training set)
preprocessors_pah = {
    "m3": fit_preprocessor(df_combined_pah, keep_cols, TARGET, "m3"),
    "no_flags": fit_preprocessor(df_combined_pah, keep_cols, TARGET, "no_flags"),
    "knn": fit_preprocessor(df_combined_pah, keep_cols, TARGET, "knn"),
    "sentinel": fit_preprocessor(df_combined_pah, keep_cols, TARGET, "sentinel"),
}

preprocessors_cftr = {
    "m3": fit_preprocessor(df_combined_cftr, keep_cols, TARGET, "m3"),
    "no_flags": fit_preprocessor(df_combined_cftr, keep_cols, TARGET, "no_flags"),
    "knn": fit_preprocessor(df_combined_cftr, keep_cols, TARGET, "knn"),
    "sentinel": fit_preprocessor(df_combined_cftr, keep_cols, TARGET, "sentinel"),
}

# Transform training & panel data for baseline (M3)
X_combined_pah_m3 = transform_X(df_combined_pah, keep_cols, preprocessors_pah["m3"])
y_combined_pah = df_combined_pah[TARGET].values

X_pah_m3 = transform_X(df_pah, keep_cols, preprocessors_pah["m3"])
y_pah = df_pah[TARGET].values

X_combined_cftr_m3 = transform_X(df_combined_cftr, keep_cols, preprocessors_cftr["m3"])
y_combined_cftr = df_combined_cftr[TARGET].values

X_cftr_m3 = transform_X(df_cftr, keep_cols, preprocessors_cftr["m3"])
y_cftr = df_cftr[TARGET].values

print(f"Preprocessing complete:")
print(f"  X_combined_pah: {X_combined_pah_m3.shape}")
print(f"  X_pah: {X_pah_m3.shape}")
print(f"  X_combined_cftr: {X_combined_cftr_m3.shape}")
print(f"  X_cftr: {X_cftr_m3.shape}")

Preprocessing complete:
  X_combined_pah: (3430, 435)
  X_pah: (369, 435)
  X_combined_cftr: (3688, 435)
  X_cftr: (111, 435)


# Cell 4: Evaluation Infrastructure — Weighted F1 + Bootstrap + Prior Shift

In [4]:
# --- 4a. Prior shift (Saerens 2002) ---
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    """Adjust posterior probabilities for class imbalance shift."""
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# --- 4b. Bootstrap %80/20 ---
def _f1_pos(y, p):
    """F1 score for positive class (pathogenic)."""
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    """Resample to 80% benign / 20% pathogenic."""
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    # Keep all negatives, downsample positives
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    pos_kept = rng.choice(pos, size=npos, replace=True)
    keep = np.concatenate([neg, pos_kept])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    """Bootstrap evaluation on %80/20 distribution."""
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {
        "mean": float(f1s.mean()),
        "std": float(f1s.std()),
        "lo": float(np.percentile(f1s, 2.5)),
        "hi": float(np.percentile(f1s, 97.5))
    }

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    """Select threshold that maximizes F1 on %80/20 distribution (N-bootstrap average)."""
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

# --- 4c. Weighted F1 (GÖREV 1) ---
def weighted_f1_score(y_true, y_pred, target_benign_ratio=FINAL_BENIGN_FRAC):
    """
    Compute F1 with sample_weight simulating target test distribution.
    All samples used (deterministic, no downsampling).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    n_total = n_pos + n_neg
    benign_frac = n_neg / n_total if n_total > 0 else 0.5
    path_frac = n_pos / n_total if n_total > 0 else 0.5
    
    if benign_frac > 0:
        w_benign = target_benign_ratio / benign_frac
    else:
        w_benign = 1.0
    
    if path_frac > 0:
        w_path = (1 - target_benign_ratio) / path_frac
    else:
        w_path = 1.0
    
    weights = np.where(y_true == 0, w_benign, w_path)
    return f1_score(y_true, y_pred, sample_weight=weights, zero_division=0)

def optimize_threshold_weighted(y_true, y_prob, target_benign_ratio=FINAL_BENIGN_FRAC):
    """Find threshold that maximizes weighted F1."""
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.arange(0.05, 0.95, 0.01):
        thr_r = round(thr, 2)
        y_pred = (y_prob >= thr_r).astype(int)
        f1_val = weighted_f1_score(y_true, y_pred, target_benign_ratio)
        if f1_val > best_f1:
            best_f1 = f1_val
            best_thr = thr_r
    return best_thr, best_f1

# --- 4d. Full evaluation (all 3 threshold methods) ---
def full_eval(y_true, prob, pi_train, label=""):
    """
    Evaluate with 3 threshold methods:
      1. Raw F1-max (train distribution)
      2. Weighted F1-max (80/20 distribution)
      3. Bootstrap 80/20 robust (NB21 method)
    """
    prob_adj = adjust_prior_shift(prob, pi_train)
    
    # Method 1: F1-max on train distribution
    thr_raw, f1_raw = optimize_threshold(y_true, prob)
    
    # Method 2: Weighted F1-max (NEW)
    thr_weighted, f1_weighted = optimize_threshold_weighted(y_true, prob, FINAL_BENIGN_FRAC)
    
    # Method 3: Bootstrap %80/20 robust (NB21)
    thr_boot = select_threshold_8020_robust(y_true, prob_adj)
    
    # Evaluate all three at bootstrap %80/20
    boot_raw = bootstrap_8020(y_true, prob_adj, thr_raw)
    boot_weighted = bootstrap_8020(y_true, prob_adj, thr_weighted)
    boot_boot = bootstrap_8020(y_true, prob_adj, thr_boot)
    
    # MCC at each threshold
    mcc_raw = matthews_corrcoef(y_true, (prob_adj >= thr_raw).astype(int))
    mcc_weighted = matthews_corrcoef(y_true, (prob_adj >= thr_weighted).astype(int))
    mcc_boot = matthews_corrcoef(y_true, (prob_adj >= thr_boot).astype(int))
    
    return {
        "label": label,
        "thr_raw": thr_raw,
        "thr_weighted": thr_weighted,
        "thr_boot": thr_boot,
        "f1_weighted_at_thr": f1_weighted,
        "boot_raw": boot_raw,
        "boot_weighted": boot_weighted,
        "boot_boot": boot_boot,
        "mcc_raw": mcc_raw,
        "mcc_weighted": mcc_weighted,
        "mcc_boot": mcc_boot,
    }

print("Evaluation infrastructure ready.")

Evaluation infrastructure ready.


# Cell 5: GÖREV 1+2 — Weighted F1 + Threshold Correction (PAH)

In [5]:
print("\n" + "="*70)
print("GÖREV 1+2: Weighted F1 + Threshold Correction (PAH)")
print("="*70)

# Baseline: NB21 P4_COMBINED_BalBag
LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

# Train P4_COMBINED_BalBag
print("\nTraining P4_COMBINED_BalBag (NB21 best)...")
bb_pah = BalancedBaggingClassifier(
    estimator=LGBMClassifier(**LGBM_PARAMS),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_pah.fit(X_combined_pah_m3, y_combined_pah)
p_pah_train = bb_pah.predict_proba(X_combined_pah_m3)[:, 1]
p_pah_panel = bb_pah.predict_proba(X_pah_m3)[:, 1]

# Evaluate with all 3 threshold methods
pi_train_pah = float(y_combined_pah.mean())
pah_eval = full_eval(y_pah, p_pah_panel, pi_train_pah, "PAH_P4_COMBINED_BalBag")

print(f"\nPAH Results (3 threshold methods):")
print(f"  Threshold (raw F1-max):     {pah_eval['thr_raw']:.4f}")
print(f"  Threshold (weighted F1-max): {pah_eval['thr_weighted']:.4f}")
print(f"  Threshold (boot %80/20):     {pah_eval['thr_boot']:.4f}")
print(f"")
print(f"  MCC: raw={pah_eval['mcc_raw']:.4f}, weighted={pah_eval['mcc_weighted']:.4f}, boot={pah_eval['mcc_boot']:.4f}")
print(f"")
print(f"  Boot %80/20 F1:")
print(f"    Raw thr:      {pah_eval['boot_raw']['mean']:.4f} +/- {pah_eval['boot_raw']['std']:.4f}")
print(f"    Weighted thr: {pah_eval['boot_weighted']['mean']:.4f} +/- {pah_eval['boot_weighted']['std']:.4f}")
print(f"    Boot thr:     {pah_eval['boot_boot']['mean']:.4f} +/- {pah_eval['boot_boot']['std']:.4f}")
print(f"\n  NB21 P4 baseline: Boot F1=0.582")

# Save comparison
pah_comparison_rows = []
for method, thr, boot_res, mcc_val in [
    ("raw_f1_max", pah_eval["thr_raw"], pah_eval["boot_raw"], pah_eval["mcc_raw"]),
    ("weighted_f1_max", pah_eval["thr_weighted"], pah_eval["boot_weighted"], pah_eval["mcc_weighted"]),
    ("boot_8020_robust", pah_eval["thr_boot"], pah_eval["boot_boot"], pah_eval["mcc_boot"]),
]:
    pah_comparison_rows.append({
        "Method": method,
        "Threshold": thr,
        "Boot_F1_mean": boot_res["mean"],
        "Boot_F1_std": boot_res["std"],
        "Boot_F1_lo": boot_res["lo"],
        "Boot_F1_hi": boot_res["hi"],
        "MCC": mcc_val,
    })

pah_comparison_df = pd.DataFrame(pah_comparison_rows)
pah_comparison_path = os.path.join(RESULTS_DIR, "g1_pah_threshold_comparison.csv")
pah_comparison_df.to_csv(pah_comparison_path, index=False)
print(f"\nSaved: {pah_comparison_path}")


GÖREV 1+2: Weighted F1 + Threshold Correction (PAH)

Training P4_COMBINED_BalBag (NB21 best)...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


PAH Results (3 threshold methods):
  Threshold (raw F1-max):     0.1600
  Threshold (weighted F1-max): 0.8800
  Threshold (boot %80/20):     0.4200

  MCC: raw=0.4969, weighted=0.1403, boot=0.4013

  Boot %80/20 F1:
    Raw thr:      0.5595 +/- 0.0440
    Weighted thr: 0.2325 +/- 0.1266
    Boot thr:     0.5903 +/- 0.0732

  NB21 P4 baseline: Boot F1=0.582

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization/g1_pah_threshold_comparison.csv


# Cell 6: GÖREV 1+2 — Weighted F1 + Threshold Correction (CFTR)

In [6]:
print("\n" + "="*70)
print("GÖREV 1+2: Weighted F1 + Threshold Correction (CFTR)")
print("="*70)

# Train S0c_COMBINED (LightGBM, no BalancedBagging for CFTR in NB20)
print("\nTraining S0c_COMBINED (NB20 best)...")
m_cftr = LGBMClassifier(**LGBM_PARAMS)
m_cftr.fit(X_combined_cftr_m3, y_combined_cftr)
p_cftr_train = m_cftr.predict_proba(X_combined_cftr_m3)[:, 1]
p_cftr_panel = m_cftr.predict_proba(X_cftr_m3)[:, 1]

# Evaluate with all 3 threshold methods
pi_train_cftr = float(y_combined_cftr.mean())
cftr_eval = full_eval(y_cftr, p_cftr_panel, pi_train_cftr, "CFTR_S0c_COMBINED")

print(f"\nCFTR Results (3 threshold methods):")
print(f"  CFTR: n={len(y_cftr)} (pos={y_cftr.sum()}, neg={(y_cftr==0).sum()})")
print(f"  Threshold (raw F1-max):     {cftr_eval['thr_raw']:.4f}")
print(f"  Threshold (weighted F1-max): {cftr_eval['thr_weighted']:.4f}")
print(f"  Threshold (boot %80/20):     {cftr_eval['thr_boot']:.4f}")
print(f"")
print(f"  MCC: raw={cftr_eval['mcc_raw']:.4f}, weighted={cftr_eval['mcc_weighted']:.4f}, boot={cftr_eval['mcc_boot']:.4f}")
print(f"")
print(f"  Boot %80/20 F1:")
print(f"    Raw thr:      {cftr_eval['boot_raw']['mean']:.4f} +/- {cftr_eval['boot_raw']['std']:.4f}")
print(f"    Weighted thr: {cftr_eval['boot_weighted']['mean']:.4f} +/- {cftr_eval['boot_weighted']['std']:.4f}")
print(f"    Boot thr:     {cftr_eval['boot_boot']['mean']:.4f} +/- {cftr_eval['boot_boot']['std']:.4f}")
print(f"\n  WARNING: CFTR n=21 benign -> CI very wide, differences likely noise")
print(f"  NB20 S0c baseline: Boot F1=0.863, FP=0")

# Save comparison
cftr_comparison_rows = []
for method, thr, boot_res, mcc_val in [
    ("raw_f1_max", cftr_eval["thr_raw"], cftr_eval["boot_raw"], cftr_eval["mcc_raw"]),
    ("weighted_f1_max", cftr_eval["thr_weighted"], cftr_eval["boot_weighted"], cftr_eval["mcc_weighted"]),
    ("boot_8020_robust", cftr_eval["thr_boot"], cftr_eval["boot_boot"], cftr_eval["mcc_boot"]),
]:
    cftr_comparison_rows.append({
        "Method": method,
        "Threshold": thr,
        "Boot_F1_mean": boot_res["mean"],
        "Boot_F1_std": boot_res["std"],
        "Boot_F1_lo": boot_res["lo"],
        "Boot_F1_hi": boot_res["hi"],
        "MCC": mcc_val,
    })

cftr_comparison_df = pd.DataFrame(cftr_comparison_rows)
cftr_comparison_path = os.path.join(RESULTS_DIR, "g2_cftr_threshold_comparison.csv")
cftr_comparison_df.to_csv(cftr_comparison_path, index=False)
print(f"\nSaved: {cftr_comparison_path}")


GÖREV 1+2: Weighted F1 + Threshold Correction (CFTR)

Training S0c_COMBINED (NB20 best)...

CFTR Results (3 threshold methods):
  CFTR: n=111 (pos=90, neg=21)
  Threshold (raw F1-max):     0.2400
  Threshold (weighted F1-max): 0.7600
  Threshold (boot %80/20):     0.2300

  MCC: raw=0.5855, weighted=0.3142, boot=0.5855

  Boot %80/20 F1:
    Raw thr:      0.7230 +/- 0.1099
    Weighted thr: 0.5456 +/- 0.2205
    Boot thr:     0.7230 +/- 0.1099

  NB20 S0c baseline: Boot F1=0.863, FP=0

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization/g2_cftr_threshold_comparison.csv


# Cell 7: GÖREV 4 — Missing Value Robustness (PAH)

In [7]:
print("\n" + "="*70)
print("GÖREV 4: Missing Value Robustness (PAH)")
print("="*70)

# 4 strategies: M3, no_flags, KNN, sentinel
pah_missing_results = []

for strategy in ["m3", "no_flags", "knn", "sentinel"]:
    print(f"\n  Strategy: {strategy}...")
    
    # Transform data with strategy
    X_train_strat = transform_X(df_combined_pah, keep_cols, preprocessors_pah[strategy])
    X_panel_strat = transform_X(df_pah, keep_cols, preprocessors_pah[strategy])
    
    # Train BalancedBagging+LGBM
    bb_strat = BalancedBaggingClassifier(
        estimator=LGBMClassifier(**LGBM_PARAMS),
        n_estimators=20,
        sampling_strategy="not minority",
        random_state=SEED,
        n_jobs=-1
    )
    bb_strat.fit(X_train_strat, y_combined_pah)
    p_panel_strat = bb_strat.predict_proba(X_panel_strat)[:, 1]
    
    # Evaluate with weighted F1 threshold — use PANEL labels (y_pah) with panel predictions
    thr_weighted, _ = optimize_threshold_weighted(y_pah, p_panel_strat, FINAL_BENIGN_FRAC)
    
    # Prior-shift adjust
    p_adj = adjust_prior_shift(p_panel_strat, pi_train_pah)
    
    # Bootstrap %80/20
    boot_res = bootstrap_8020(y_pah, p_adj, thr_weighted)
    
    # Metrics
    y_pred = (p_adj >= thr_weighted).astype(int)
    mcc = matthews_corrcoef(y_pah, y_pred)
    precision = precision_score(y_pah, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_pah, y_pred, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_pah, y_pred, labels=[0, 1]).ravel()
    
    print(f"    Threshold: {thr_weighted:.4f}")
    print(f"    Boot F1: {boot_res['mean']:.4f} +/- {boot_res['std']:.4f}")
    print(f"    MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
    
    pah_missing_results.append({
        "Strategy": strategy,
        "Threshold": thr_weighted,
        "Boot_F1_mean": boot_res["mean"],
        "Boot_F1_std": boot_res["std"],
        "Boot_F1_lo": boot_res["lo"],
        "Boot_F1_hi": boot_res["hi"],
        "MCC": mcc,
        "Precision": precision,
        "Recall": recall,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

pah_missing_df = pd.DataFrame(pah_missing_results)
pah_missing_path = os.path.join(RESULTS_DIR, "g4_pah_missing_robustness.csv")
pah_missing_df.to_csv(pah_missing_path, index=False)
print(f"\nSaved: {pah_missing_path}")

print(f"\n  Best strategy (Boot F1): {pah_missing_df.loc[pah_missing_df['Boot_F1_mean'].idxmax(), 'Strategy']}")
print(f"  NB21 M3 baseline: Boot F1=0.582")


GÖREV 4: Missing Value Robustness (PAH)

  Strategy: m3...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

    Threshold: 0.8800
    Boot F1: 0.2325 +/- 0.1266
    MCC: 0.1403, Precision: 0.9615, Recall: 0.1629

  Strategy: no_flags...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

    Threshold: 0.8800
    Boot F1: 0.2325 +/- 0.1266
    MCC: 0.1403, Precision: 0.9615, Recall: 0.1629

  Strategy: knn...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

    Threshold: 0.9200
    Boot F1: 0.1558 +/- 0.1147
    MCC: 0.1181, Precision: 0.9706, Recall: 0.1075

  Strategy: sentinel...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

    Threshold: 0.6400
    Boot F1: 0.4565 +/- 0.1383
    MCC: 0.2521, Precision: 0.9669, Recall: 0.3811

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization/g4_pah_missing_robustness.csv

  Best strategy (Boot F1): sentinel
  NB21 M3 baseline: Boot F1=0.582


# Cell 8: GÖREV 4 — Missing Value Robustness (CFTR)

In [8]:
print("\n" + "="*70)
print("GÖREV 4: Missing Value Robustness (CFTR)")
print("="*70)

cftr_missing_results = []

for strategy in ["m3", "no_flags", "knn", "sentinel"]:
    print(f"\n  Strategy: {strategy}...")
    
    # Transform data with strategy
    X_train_strat = transform_X(df_combined_cftr, keep_cols, preprocessors_cftr[strategy])
    X_panel_strat = transform_X(df_cftr, keep_cols, preprocessors_cftr[strategy])
    
    # Train LightGBM (no BalancedBagging for baseline consistency)
    m_strat = LGBMClassifier(**LGBM_PARAMS)
    m_strat.fit(X_train_strat, y_combined_cftr)
    p_panel_strat = m_strat.predict_proba(X_panel_strat)[:, 1]
    
    # Evaluate with weighted F1 threshold — use PANEL labels (y_cftr) with panel predictions
    thr_weighted, _ = optimize_threshold_weighted(y_cftr, p_panel_strat, FINAL_BENIGN_FRAC)
    
    # Prior-shift adjust
    p_adj = adjust_prior_shift(p_panel_strat, pi_train_cftr)
    
    # Bootstrap %80/20
    boot_res = bootstrap_8020(y_cftr, p_adj, thr_weighted)
    
    # Metrics
    y_pred = (p_adj >= thr_weighted).astype(int)
    mcc = matthews_corrcoef(y_cftr, y_pred)
    precision = precision_score(y_cftr, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_cftr, y_pred, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_cftr, y_pred, labels=[0, 1]).ravel()
    
    print(f"    Threshold: {thr_weighted:.4f}")
    print(f"    Boot F1: {boot_res['mean']:.4f} +/- {boot_res['std']:.4f}")
    print(f"    MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
    print(f"    Confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    
    cftr_missing_results.append({
        "Strategy": strategy,
        "Threshold": thr_weighted,
        "Boot_F1_mean": boot_res["mean"],
        "Boot_F1_std": boot_res["std"],
        "Boot_F1_lo": boot_res["lo"],
        "Boot_F1_hi": boot_res["hi"],
        "MCC": mcc,
        "Precision": precision,
        "Recall": recall,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

cftr_missing_df = pd.DataFrame(cftr_missing_results)
cftr_missing_path = os.path.join(RESULTS_DIR, "g4_cftr_missing_robustness.csv")
cftr_missing_df.to_csv(cftr_missing_path, index=False)
print(f"\nSaved: {cftr_missing_path}")

print(f"\n  Best strategy (Boot F1): {cftr_missing_df.loc[cftr_missing_df['Boot_F1_mean'].idxmax(), 'Strategy']}")
print(f"  WARNING: CFTR n=21 benign, CI very wide")
print(f"  NB20 M3 baseline: Boot F1=0.863, FP=0")


GÖREV 4: Missing Value Robustness (CFTR)

  Strategy: m3...
    Threshold: 0.7600
    Boot F1: 0.5456 +/- 0.2205
    MCC: 0.3142, Precision: 1.0000, Recall: 0.3667
    Confusion: TN=21, FP=0, FN=57, TP=33

  Strategy: no_flags...
    Threshold: 0.7600
    Boot F1: 0.5456 +/- 0.2205
    MCC: 0.3142, Precision: 1.0000, Recall: 0.3667
    Confusion: TN=21, FP=0, FN=57, TP=33

  Strategy: knn...
    Threshold: 0.8600
    Boot F1: 0.3906 +/- 0.2584
    MCC: 0.2537, Precision: 1.0000, Recall: 0.2667
    Confusion: TN=21, FP=0, FN=66, TP=24

  Strategy: sentinel...
    Threshold: 0.8900
    Boot F1: 0.2579 +/- 0.2241
    MCC: 0.1835, Precision: 1.0000, Recall: 0.1556
    Confusion: TN=21, FP=0, FN=76, TP=14

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization/g4_cftr_missing_robustness.csv

  Best strategy (Boot F1): m3
  NB20 M3 baseline: Boot F1=0.863, FP=0


# Cell 9: GÖREV 3 — Label Noise Detection (PAH only)

In [9]:
print("\n" + "="*70)
print("GÖREV 3: Label Noise Detection (PAH)")
print("="*70)

# Cross-model consensus: 3 models with different inductive biases
models_for_noise = {
    'LightGBM': LGBMClassifier(**LGBM_PARAMS),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        use_label_encoder=False, eval_metric='logloss',
        random_state=SEED, verbosity=0, n_jobs=-1
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        random_seed=SEED, verbose=0
    ),
}

print("\nCross-model consensus error analysis...")
errors_per_sample = np.zeros(len(y_pah))
all_proba = {}

for name, model in models_for_noise.items():
    print(f"  Training {name}...")
    model.fit(X_combined_pah_m3, y_combined_pah)
    p = model.predict_proba(X_pah_m3)[:, 1]
    all_proba[name] = p
    
    # Use weighted F1 threshold — panel labels (y_pah) with panel predictions (p)
    thr, _ = optimize_threshold_weighted(y_pah, p, FINAL_BENIGN_FRAC)
    pred = (p >= thr).astype(int)
    
    # Count errors
    errors = (pred != y_pah).astype(int)
    errors_per_sample += errors
    
    wrong = (pred != y_pah).sum()
    print(f"    Wrong predictions: {wrong}/{len(y_pah)}")

# Suspicious samples: all 3 models agree they're wrong
n_models = len(models_for_noise)
suspicious_mask = errors_per_sample == n_models
suspicious_idx = np.where(suspicious_mask)[0]

print(f"\nSuspicious samples (all {n_models} models wrong): {len(suspicious_idx)}/{len(y_pah)}")
if len(suspicious_idx) > 0:
    sus_labels = y_pah[suspicious_idx]
    print(f"  Label distribution: pos={sus_labels.sum()}, neg={(sus_labels==0).sum()}")
    print(f"  Indices: {suspicious_idx[:10]}..." if len(suspicious_idx) > 10 else f"  Indices: {suspicious_idx}")

# Try cleanlab if available
cleanlab_issues = None
if HAS_CLEANLAB:
    print("\n  Running cleanlab (OOF-based)...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_proba = np.zeros(len(y_pah))
    
    for tri, vai in skf.split(X_pah_m3, y_pah):
        m = LGBMClassifier(**LGBM_PARAMS)
        m.fit(X_pah_m3.iloc[tri], y_pah[tri])
        oof_proba[vai] = m.predict_proba(X_pah_m3.iloc[vai])[:, 1]
    
    pred_probs = np.column_stack([1 - oof_proba, oof_proba])
    try:
        cleanlab_issues = find_label_issues(y_pah, pred_probs, return_indices_ranked_by='self_confidence')
        print(f"  Cleanlab found {len(cleanlab_issues)} potential label issues")
    except Exception as e:
        print(f"  Cleanlab error: {e}")
else:
    print("\n  Cleanlab not available (pip install cleanlab)")

# Save results
noise_results = []
for idx in range(len(y_pah)):
    row = {
        "PAH_idx": idx,
        "Variant_ID": df_pah.iloc[idx][ID_COL] if ID_COL in df_pah.columns else "",
        "True_label": int(y_pah[idx]),
        "Model_errors": int(errors_per_sample[idx]),
        "Suspicious": int(suspicious_mask[idx]),
    }
    for name in all_proba:
        row[f"Proba_{name}"] = float(all_proba[name][idx])
    noise_results.append(row)

noise_df = pd.DataFrame(noise_results)
noise_path = os.path.join(RESULTS_DIR, "g3_pah_label_noise.csv")
noise_df.to_csv(noise_path, index=False)
print(f"\nSaved: {noise_path}")


GÖREV 3: Label Noise Detection (PAH)

Cross-model consensus error analysis...
  Training LightGBM...
    Wrong predictions: 79/369
  Training XGBoost...
    Wrong predictions: 70/369
  Training CatBoost...
    Wrong predictions: 74/369

Suspicious samples (all 3 models wrong): 49/369
  Label distribution: pos=36, neg=13
  Indices: [ 9 25 33 38 46 53 58 63 77 84]...

  Running cleanlab (OOF-based)...
  Cleanlab found 48 potential label issues

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization/g3_pah_label_noise.csv


# Cell 10: Results Compilation & Summary

In [10]:
print("\n" + "="*80)
print("NB27 RESULTS SUMMARY")
print("="*80)

# GÖREV 1+2 Comparison Table
print("\n--- GÖREV 1+2: Threshold Methods Comparison ---\n")
print("PAH:")
print(pah_comparison_df.to_string(index=False))
print("\nCFTR:")
print(cftr_comparison_df.to_string(index=False))

# GÖREV 4 Missing Robustness
print("\n--- GÖREV 4: Missing Value Robustness ---\n")
print("PAH:")
print(pah_missing_df[["Strategy", "Boot_F1_mean", "Boot_F1_std", "MCC", "Precision", "Recall"]].to_string(index=False))
print("\nCFTR:")
print(cftr_missing_df[["Strategy", "Boot_F1_mean", "Boot_F1_std", "MCC", "Precision", "Recall"]].to_string(index=False))

# GÖREV 3 Label Noise
print("\n--- GÖREV 3: Label Noise Detection (PAH) ---")
print(f"Total suspicious samples (all 3 models wrong): {(noise_df['Suspicious'] == 1).sum()}")
if (noise_df['Suspicious'] == 1).sum() > 0:
    sus_rows = noise_df[noise_df['Suspicious'] == 1]
    print(f"Suspicious labels: pos={(sus_rows['True_label'] == 1).sum()}, neg={(sus_rows['True_label'] == 0).sum()}")
    print(f"\nTop 5 most suspicious (highest model disagreement):")
    print(sus_rows.nlargest(5, "Model_errors")[["PAH_idx", "True_label", "Model_errors", "Proba_LightGBM", "Proba_XGBoost", "Proba_CatBoost"]].to_string(index=False))

print("\n" + "="*80)
print("All results saved to: " + RESULTS_DIR)
print("="*80)


NB27 RESULTS SUMMARY

--- GÖREV 1+2: Threshold Methods Comparison ---

PAH:
          Method  Threshold  Boot_F1_mean  Boot_F1_std  Boot_F1_lo  Boot_F1_hi      MCC
      raw_f1_max       0.16      0.559510     0.043952    0.500000    0.625000 0.496889
 weighted_f1_max       0.88      0.232511     0.126642    0.111111    0.454545 0.140348
boot_8020_robust       0.42      0.590253     0.073199    0.448153    0.684211 0.401263

CFTR:
          Method  Threshold  Boot_F1_mean  Boot_F1_std  Boot_F1_lo  Boot_F1_hi      MCC
      raw_f1_max       0.24      0.723030     0.109867    0.444444    0.833333 0.585540
 weighted_f1_max       0.76      0.545556     0.220492    0.000000    0.857639 0.314194
boot_8020_robust       0.23      0.723030     0.109867    0.444444    0.833333 0.585540

--- GÖREV 4: Missing Value Robustness ---

PAH:
Strategy  Boot_F1_mean  Boot_F1_std      MCC  Precision   Recall
      m3      0.232511     0.126642 0.140348   0.961538 0.162866
no_flags      0.232511     0.1266

# Cell 11: Visualizations

In [11]:
# --- Figure 1: Threshold Sweep (PAH weighted F1 vs threshold) ---
print("\nGenerating visualizations...")

thresholds = np.arange(0.05, 0.95, 0.01)
weighted_f1s = []
for thr in thresholds:
    y_pred = (p_pah_panel >= thr).astype(int)
    f1 = weighted_f1_score(y_pah, y_pred, FINAL_BENIGN_FRAC)
    weighted_f1s.append(f1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, weighted_f1s, 'b-', linewidth=2, label="Weighted F1")
best_idx = np.argmax(weighted_f1s)
ax.axvline(thresholds[best_idx], color='r', linestyle='--', alpha=0.7, label=f"Optimal thr={thresholds[best_idx]:.3f}")
ax.axvline(pah_eval['thr_raw'], color='g', linestyle=':', alpha=0.7, label=f"Raw F1-max thr={pah_eval['thr_raw']:.3f}")
ax.axvline(pah_eval['thr_boot'], color='orange', linestyle='-.', alpha=0.7, label=f"Boot robust thr={pah_eval['thr_boot']:.3f}")
ax.set_xlabel("Threshold", fontsize=12)
ax.set_ylabel("Weighted F1 Score", fontsize=12)
ax.set_title("NB27 — PAH: Weighted F1 vs Threshold", fontsize=14)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_pah_threshold_sweep.png"), dpi=150)
plt.close()
print("  Saved: fig1_pah_threshold_sweep.png")

# --- Figure 2: Missing Robustness (PAH) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

strategies = pah_missing_df["Strategy"].tolist()
boot_means = pah_missing_df["Boot_F1_mean"].tolist()
boot_stds = pah_missing_df["Boot_F1_std"].tolist()

colors = plt.cm.Set3(np.linspace(0, 1, len(strategies)))
bars1 = ax1.bar(strategies, boot_means, yerr=boot_stds, capsize=5, color=colors, alpha=0.8)
ax1.axhline(y=0.582, color='r', linestyle='--', alpha=0.5, label="NB21 M3 baseline (0.582)")
ax1.set_ylabel("Boot %80/20 F1", fontsize=11)
ax1.set_title("PAH: Missing Value Strategies", fontsize=12)
ax1.legend(fontsize=9)
ax1.set_ylim([0.3, 0.7])
for b in bars1:
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2, h, f"{h:.3f}", ha="center", va="bottom", fontsize=9)

# Bar for MCC
mccs = pah_missing_df["MCC"].tolist()
bars2 = ax2.bar(strategies, mccs, color=colors, alpha=0.8)
ax2.set_ylabel("MCC", fontsize=11)
ax2.set_title("PAH: MCC Comparison", fontsize=12)
ax2.set_ylim([-0.1, 0.8])
for b in bars2:
    h = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2, h, f"{h:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_pah_missing_robustness.png"), dpi=150)
plt.close()
print("  Saved: fig2_pah_missing_robustness.png")

# --- Figure 3: Missing Robustness (CFTR) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

strategies_cftr = cftr_missing_df["Strategy"].tolist()
boot_means_cftr = cftr_missing_df["Boot_F1_mean"].tolist()
boot_stds_cftr = cftr_missing_df["Boot_F1_std"].tolist()

colors_cftr = plt.cm.Set3(np.linspace(0, 1, len(strategies_cftr)))
bars1 = ax1.bar(strategies_cftr, boot_means_cftr, yerr=boot_stds_cftr, capsize=5, color=colors_cftr, alpha=0.8)
ax1.axhline(y=0.863, color='r', linestyle='--', alpha=0.5, label="NB20 M3 baseline (0.863)")
ax1.set_ylabel("Boot %80/20 F1", fontsize=11)
ax1.set_title("CFTR: Missing Value Strategies (n=21 benign → wide CI)", fontsize=12)
ax1.legend(fontsize=9)
for b in bars1:
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2, h, f"{h:.3f}", ha="center", va="bottom", fontsize=9)

# Bar for MCC
mccs_cftr = cftr_missing_df["MCC"].tolist()
bars2 = ax2.bar(strategies_cftr, mccs_cftr, color=colors_cftr, alpha=0.8)
ax2.set_ylabel("MCC", fontsize=11)
ax2.set_title("CFTR: MCC Comparison", fontsize=12)
for b in bars2:
    h = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2, h, f"{h:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_cftr_missing_robustness.png"), dpi=150)
plt.close()
print("  Saved: fig3_cftr_missing_robustness.png")

# --- Figure 4: Label Noise Heatmap (PAH) ---
# Show top 20 most suspicious samples + models agreement
suspicious_full = noise_df.sort_values("Model_errors", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
# Create heatmap: samples × models, color = prediction agreement
model_names = list(models_for_noise.keys())
heatmap_data = np.zeros((len(suspicious_full), len(model_names)))

for i, (_, row) in enumerate(suspicious_full.iterrows()):
    idx = int(row["PAH_idx"])
    true_label = int(row["True_label"])
    for j, name in enumerate(model_names):
        proba = row[f"Proba_{name}"]
        pred = 1 if proba >= 0.5 else 0
        # 0 = correct, 1 = wrong
        heatmap_data[i, j] = 0 if pred == true_label else 1

im = ax.imshow(heatmap_data, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(np.arange(len(model_names)))
ax.set_yticks(np.arange(len(suspicious_full)))
ax.set_xticklabels(model_names, fontsize=10)
ax.set_yticklabels([f"Sample {int(row['PAH_idx'])}" for _, row in suspicious_full.iterrows()], fontsize=8)
ax.set_xlabel("Model", fontsize=11)
ax.set_ylabel("Sample Index (sorted by error count)", fontsize=11)
ax.set_title("NB27 — PAH: Cross-Model Error Agreement (Top 20 suspicious)\nRed=wrong, Green=correct", fontsize=12)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Prediction Error", fontsize=10)

# Add text
for i in range(len(suspicious_full)):
    for j in range(len(model_names)):
        text = ax.text(j, i, "X" if heatmap_data[i, j] > 0.5 else "✓",
                       ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_pah_label_noise_heatmap.png"), dpi=150)
plt.close()
print("  Saved: fig4_pah_label_noise_heatmap.png")

print("\nAll visualizations complete.")


Generating visualizations...
  Saved: fig1_pah_threshold_sweep.png
  Saved: fig2_pah_missing_robustness.png
  Saved: fig3_cftr_missing_robustness.png
  Saved: fig4_pah_label_noise_heatmap.png

All visualizations complete.


# Cell 12: PDF Report Summary

In [14]:
from fpdf import FPDF
UNICODE_FONT_PATH = "/Library/Fonts/Arial Unicode.ttf"

class OptimizationReport(FPDF):
    def __init__(self):
        super().__init__()
        self.add_font("ArialUni", "", UNICODE_FONT_PATH)
        self.add_font("ArialUni", "B", UNICODE_FONT_PATH)
        self.add_font("ArialUni", "I", UNICODE_FONT_PATH)

    def header(self):
        self.set_font("ArialUni", "B", 11)
        self.cell(0, 8, "NB27 Optimization Report | TEKNOFEST 2026", 0, 1, "C")
        self.set_draw_color(0, 102, 153)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font("ArialUni", "I", 8)
        self.cell(0, 10, f"Page {self.page_no()}/{{nb}}", 0, 0, "C")

    def section(self, title):
        self.set_font("ArialUni", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body(self, text):
        self.set_font("ArialUni", "", 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

pdf = OptimizationReport()
pdf.alias_nb_pages()
pdf.add_page()

# Title
pdf.set_font("ArialUni", "B", 16)
pdf.cell(0, 15, "NB27 Optimization Report", 0, 1, "C")
pdf.set_font("ArialUni", "", 11)
pdf.cell(0, 8, "Weighted F1 + Threshold + Missing Robustness + Label Noise", 0, 1, "C")
pdf.cell(0, 6, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}", 0, 1, "C")
pdf.ln(5)

# Executive Summary
pdf.section("Executive Summary")
pdf.body(
    f"NB27 implements 4 optimization GÖREVs from to-do.md:\n"
    f"1. Ağırlıklı F1 (weighted): Deterministic evaluation on 80/20 distribution\n"
    f"2. Threshold correction: Compare raw F1-max, weighted F1-max, bootstrap robust\n"
    f"3. Label noise: Cross-model consensus (3 models) + cleanlab (if available)\n"     
    f"4. Missing robustness: 4 strategies (M3, no-flags, KNN, sentinel)\n"
    f"\n"
    f"PAH baseline: NB21 P4_COMBINED_BalBag Boot F1=0.582\n"
    f"CFTR baseline: NB20 S0c_COMBINED Boot F1=0.863"
)
# GÖREV 1+2 Results
pdf.section("GÖREV 1+2: Threshold Methods Comparison")
pdf.body("Three threshold selection strategies evaluated on PAH and CFTR:\n")
pdf.set_font("ArialUni", "B", 10)
pdf.cell(0, 6, "PAH Results", 0, 1)
pdf.set_font("ArialUni", "", 8)
pdf.ln(1)

headers = ["Method", "Thr", "F1 Boot", "±Std", "MCC"]
col_widths = [50, 20, 30, 25, 25]
for i, h in enumerate(headers):
    pdf.cell(col_widths[i], 5, h, 1, 0, "C")
pdf.ln()

for _, row in pah_comparison_df.iterrows():
    pdf.cell(col_widths[0], 5, row["Method"][:20], 1, 0, "L")
    pdf.cell(col_widths[1], 5, f"{row['Threshold']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[2], 5, f"{row['Boot_F1_mean']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[3], 5, f"{row['Boot_F1_std']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[4], 5, f"{row['MCC']:.3f}", 1, 0, "C")
    pdf.ln()

pdf.ln(3)
pdf.set_font("ArialUni", "B", 10)
pdf.cell(0, 6, "CFTR Results (WARNING: n=21 benign → wide CI)", 0, 1)
pdf.set_font("ArialUni", "", 8)
pdf.ln(1)

for i, h in enumerate(headers):
    pdf.cell(col_widths[i], 5, h, 1, 0, "C")
pdf.ln()

for _, row in cftr_comparison_df.iterrows():
    pdf.cell(col_widths[0], 5, row["Method"][:20], 1, 0, "L")
    pdf.cell(col_widths[1], 5, f"{row['Threshold']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[2], 5, f"{row['Boot_F1_mean']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[3], 5, f"{row['Boot_F1_std']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[4], 5, f"{row['MCC']:.3f}", 1, 0, "C")
    pdf.ln()

# Add page
pdf.add_page()

# GÖREV 4 Results
pdf.section("GÖREV 4: Missing Value Robustness")
pdf.body("4 imputation strategies tested: M3 (baseline), no-flags, KNN, sentinel\n")

pdf.set_font("ArialUni", "B", 10)
pdf.cell(0, 6, "PAH Missing Robustness", 0, 1)
pdf.set_font("ArialUni", "", 8)
pdf.ln(1)

headers = ["Strategy", "F1 Boot", "MCC", "Prec", "Rec"]
col_widths = [40, 30, 25, 25, 25]
for i, h in enumerate(headers):
    pdf.cell(col_widths[i], 5, h, 1, 0, "C")
pdf.ln()

for _, row in pah_missing_df.iterrows():
    pdf.cell(col_widths[0], 5, row["Strategy"], 1, 0, "L")
    pdf.cell(col_widths[1], 5, f"{row['Boot_F1_mean']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[2], 5, f"{row['MCC']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[3], 5, f"{row['Precision']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[4], 5, f"{row['Recall']:.3f}", 1, 0, "C")
    pdf.ln()
    
pdf.ln(3)
pdf.set_font("ArialUni", "B", 10)
pdf.cell(0, 6, "CFTR Missing Robustness", 0, 1)
pdf.set_font("ArialUni", "", 8)
pdf.ln(1)

for i, h in enumerate(headers):
    pdf.cell(col_widths[i], 5, h, 1, 0, "C")
pdf.ln()

for _, row in cftr_missing_df.iterrows():
    pdf.cell(col_widths[0], 5, row["Strategy"], 1, 0, "L")
    pdf.cell(col_widths[1], 5, f"{row['Boot_F1_mean']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[2], 5, f"{row['MCC']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[3], 5, f"{row['Precision']:.3f}", 1, 0, "C")
    pdf.cell(col_widths[4], 5, f"{row['Recall']:.3f}", 1, 0, "C")
    pdf.ln()

# Add page
pdf.add_page()

# GÖREV 3 Results
pdf.section("GÖREV 3: Label Noise Detection (PAH)")
suspicious_count = (noise_df['Suspicious'] == 1).sum()
pdf.body(
    f"Cross-model consensus (3 models: LightGBM, XGBoost, CatBoost)\n"
    f"Suspicious samples (all 3 models wrong): {suspicious_count}/{len(y_pah)}\n\n"
    f"Label distribution of suspicious samples:\n"
)
if suspicious_count > 0:
    sus_rows = noise_df[noise_df['Suspicious'] == 1]
    pos_sus = (sus_rows['True_label'] == 1).sum()
    neg_sus = (sus_rows['True_label'] == 0).sum()
    pdf.body(f"  Pathogenic: {pos_sus}, Benign: {neg_sus}")
else:
    pdf.body("No samples flagged as suspicious by all 3 models.")

# Visualizations
pdf.add_page()
pdf.section("Visualizations")

if os.path.exists(os.path.join(RESULTS_DIR, "fig1_pah_threshold_sweep.png")):
    pdf.cell(0, 6, "Figure 1: PAH Weighted F1 vs Threshold Sweep", 0, 1)
    pdf.image(os.path.join(RESULTS_DIR, "fig1_pah_threshold_sweep.png"), w=170)
    pdf.ln(3)
    if os.path.exists(os.path.join(RESULTS_DIR, "fig2_pah_missing_robustness.png")):
        pdf.cell(0, 6, "Figure 2: PAH Missing Robustness", 0, 1)
        pdf.image(os.path.join(RESULTS_DIR, "fig2_pah_missing_robustness.png"), w=170)
        pdf.ln(3)

pdf.add_page()
if os.path.exists(os.path.join(RESULTS_DIR, "fig3_cftr_missing_robustness.png")):
    pdf.cell(0, 6, "Figure 3: CFTR Missing Robustness", 0, 1)
    pdf.image(os.path.join(RESULTS_DIR, "fig3_cftr_missing_robustness.png"), w=170)
    pdf.ln(3)

if os.path.exists(os.path.join(RESULTS_DIR, "fig4_pah_label_noise_heatmap.png")):
    pdf.cell(0, 6, "Figure 4: PAH Label Noise Cross-Model Heatmap", 0, 1)
    pdf.image(os.path.join(RESULTS_DIR, "fig4_pah_label_noise_heatmap.png"), w=170)
    pdf.ln(3)

# Save PDF
pdf_path = os.path.join(REPORTS_DIR, "NB27_optimization_report.pdf")
pdf.output(pdf_path)
print(f"\nPDF report saved: {pdf_path}")

# Final summary
print("\n" + "="*80)
print("NB27 COMPLETE")
print("="*80)
print(f"\nResults saved to: {RESULTS_DIR}")
print(f"\nFiles generated:")
print(f"  - g1_pah_threshold_comparison.csv")
print(f"  - g2_cftr_threshold_comparison.csv")
print(f"  - g3_pah_label_noise.csv")
print(f"  - g4_pah_missing_robustness.csv")
print(f"  - g4_cftr_missing_robustness.csv")
print(f"  - fig1_pah_threshold_sweep.png")
print(f"  - fig2_pah_missing_robustness.png")
print(f"  - fig3_cftr_missing_robustness.png")
print(f"  - fig4_pah_label_noise_heatmap.png")
print(f"  - NB27_optimization_report.pdf")
print("\nAll experiments complete.")


PDF report saved: /Users/tefe/teknofest_model/teknofest_model/reports/NB27_optimization_report.pdf

NB27 COMPLETE

Results saved to: /Users/tefe/teknofest_model/teknofest_model/results/v12_optimization

Files generated:
  - g1_pah_threshold_comparison.csv
  - g2_cftr_threshold_comparison.csv
  - g3_pah_label_noise.csv
  - g4_pah_missing_robustness.csv
  - g4_cftr_missing_robustness.csv
  - fig1_pah_threshold_sweep.png
  - fig2_pah_missing_robustness.png
  - fig3_cftr_missing_robustness.png
  - fig4_pah_label_noise_heatmap.png
  - NB27_optimization_report.pdf

All experiments complete.
